In [7]:
import pickle
import numpy as np
import re
from pathlib import Path
from tqdm import tqdm

def combine_into_single_file(root_path: Path, output_dir: Path):
    """
    지정된 경로의 모든 시퀀스 데이터를 pkl에서 npz로 변환하여
    지정된 출력 폴더에 저장합니다.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if not root_path.is_dir():
        print(f"Error: The path {root_path} is not a valid directory.")
        return

    sequence_paths = [p for p in root_path.iterdir() if p.is_dir() and p.name != output_dir.name]

    for seq_path in tqdm(sequence_paths, desc="Processing Sequences"):
        # --- [수정된 부분 START] ---
        # 정규식을 수정하여 언더스코어(_) 사이에 있는 숫자를 찾습니다.
        match = re.search(r'_(\d+)_', seq_path.name)
        if not match:
            print(f"Warning: Could not find a valid ID pattern (e.g., '_010_') in '{seq_path.name}'. Skipping.")
            continue
        
        # match.group(1)을 사용해 괄호 안의 숫자 그룹만 가져옵니다.
        pkl_filename = f"{match.group(1)}.pkl"
        # --- [수정된 부분 END] ---
        
        data_accumulator = {}
        frame_paths = sorted([p for p in seq_path.iterdir() if p.is_dir()])

        if not frame_paths:
            print(f"Warning: Skipping empty sequence directory {seq_path.name}")
            continue

        for frame_path in frame_paths:
            pkl_file = frame_path / pkl_filename

            if pkl_file.exists():
                with open(pkl_file, 'rb') as f:
                    data = pickle.load(f)
                
                if not data_accumulator:
                    for key in data.keys():
                        data_accumulator[key] = []
                
                for key, value in data.items():
                    if key in data_accumulator:
                        data_accumulator[key].append(value)
        
        if not data_accumulator:
            print(f"Warning: No valid '{pkl_filename}' files found in {seq_path.name}")
            continue

        final_data = {}
        for key, value_list in data_accumulator.items():
            stacked_array = np.array(value_list).squeeze(axis=1)
            final_data[key] = stacked_array
            
        output_npz_path = output_dir / f"{seq_path.name}.npz"
        np.savez(output_npz_path, **final_data)

    print(f"\n✅ All sequences have been processed and saved to: {output_dir}")

if __name__ == '__main__':
    # 데이터셋의 루트 경로
    dataset_root = Path("/home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/val")
    
    # NPZ 파일이 저장될 위치
    output_location = dataset_root / "combined"
    
    combine_into_single_file(dataset_root, output_location)

Processing Sequences: 100%|██████████| 28/28 [00:01<00:00, 19.41it/s]


✅ All sequences have been processed and saved to: /home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/val/combined


In [2]:
import pickle

# 확인할 파일의 경로를 지정합니다.
file_path = '/home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/test/Gym_010_cooking1/00006/010.pkl'

try:
    # 파일을 바이너리 읽기 모드('rb')로 엽니다.
    with open(file_path, 'rb') as f:
        # pickle.load()를 사용해 파일의 데이터를 불러옵니다.
        data = pickle.load(f)

    # 데이터가 딕셔너리(dictionary) 형태일 경우, keys()를 사용해 키 값을 출력합니다.
    if isinstance(data, dict):
        print(f"'{file_path}' 파일의 키(Keys):")
        for key in data.keys():
            print(f"- {key}")
        print(data['body_pose'].shape)
    else:
        # 딕셔너리가 아닐 경우, 데이터의 타입을 출력합니다.
        print(f"해당 파일은 딕셔너리가 아니며, 데이터 타입은 '{type(data)}'입니다.")
        print("데이터 내용:", data)

except FileNotFoundError:
    print(f"에러: 파일을 찾을 수 없습니다. 경로를 확인하세요: {file_path}")
except Exception as e:
    print(f"파일을 읽는 중 에러가 발생했습니다: {e}")

'/home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/test/Gym_010_cooking1/00006/010.pkl' 파일의 키(Keys):
- betas
- global_orient
- transl
- left_hand_pose
- right_hand_pose
- jaw_pose
- leye_pose
- reye_pose
- expression
- pose_embedding
- body_pose
(1, 63)


In [4]:
import numpy as np

data = np.load("/home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/test/combined/Gym_010_cooking1.npz")
print(list(data.keys()))
print(data['global_orient'].shape)

['betas', 'global_orient', 'transl', 'left_hand_pose', 'right_hand_pose', 'jaw_pose', 'leye_pose', 'reye_pose', 'expression', 'pose_embedding', 'body_pose']
(776, 3)


In [1]:
from pathlib import Path
import h5py

def inspect_hdf5_structure(file_path: Path):
    """
    HDF5 파일의 전체 구조(그룹, 데이터셋, shape, dtype)를 출력합니다.
    """
    # 1. 파일 존재 여부 확인
    if not file_path.exists():
        print(f"Error: File not found at {file_path}")
        return

    print(f"Inspecting HDF5 file: {file_path.name}\n" + "="*50)

    try:
        # 2. HDF5 파일을 읽기 모드('r')로 열기
        with h5py.File(file_path, 'r') as f:
            
            if not f.keys():
                print("File is empty or contains no top-level groups.")
                return

            # 3. 파일의 모든 최상위 그룹(시퀀스) 순회
            for group_name in sorted(f.keys()):
                print(f"\n📁 Group: {group_name}")
                group = f[group_name]
                
                if not isinstance(group, h5py.Group) or not group.keys():
                    print("  -> This is not a group or the group is empty.")
                    continue

                # 4. 각 그룹 내부의 모든 데이터셋(텐서) 순회
                for dataset_name in sorted(group.keys()):
                    dataset = group[dataset_name]
                    
                    if isinstance(dataset, h5py.Dataset):
                        # 5. 데이터셋의 이름, shape, 데이터 타입을 출력
                        print(f"  - 📊 Dataset: {dataset_name:<20} | Shape: {str(dataset.shape):<20} | Dtype: {dataset.dtype}")
                    else:
                        print(f"  - Item: {dataset_name} (Not a Dataset)")

    except Exception as e:
        print(f"\nAn error occurred while reading the file: {e}")

    print("\n" + "="*50 + "\nInspection finished.")


if __name__ == "__main__":
    # 검사할 HDF5 파일의 경로를 지정합니다.
    hdf5_file_path = Path("/home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/rich_dataset.hdf5")
    
    inspect_hdf5_structure(hdf5_file_path)

Inspecting HDF5 file: rich_dataset.hdf5

📁 Group: Gym_010_cooking1.npz
  - 📊 Dataset: T_cpf_tm1_cpf_t      | Shape: (775, 7)             | Dtype: float32
  - 📊 Dataset: T_world_cpf          | Shape: (775, 7)             | Dtype: float32
  - 📊 Dataset: T_world_root         | Shape: (775, 7)             | Dtype: float32
  - 📊 Dataset: betas                | Shape: (1, 16)              | Dtype: float32
  - 📊 Dataset: body_quats           | Shape: (775, 21, 4)         | Dtype: float32
  - 📊 Dataset: contacts             | Shape: (775, 21)            | Dtype: float32
  - 📊 Dataset: height_from_floor    | Shape: (775, 1)             | Dtype: float32

📁 Group: Gym_010_cooking1_reflect.npz
  - 📊 Dataset: T_cpf_tm1_cpf_t      | Shape: (775, 7)             | Dtype: float32
  - 📊 Dataset: T_world_cpf          | Shape: (775, 7)             | Dtype: float32
  - 📊 Dataset: T_world_root         | Shape: (775, 7)             | Dtype: float32
  - 📊 Dataset: betas                | Shape: (1, 16)        

In [6]:
import joblib

d = joblib.load('/home/dev2/Drive_C/KHW/_egotext/head2motion/dataset/rich/WHAM/rich_test_vit.pth')
print(d['betas'][0].shape)
print(d.keys())


torch.Size([776, 10])
dict_keys(['bbox', 'gender', 'res', 'vid', 'pose', 'betas', 'kp2d', 'frame_id', 'cam_poses', 'features', 'flipped_features', 'flipped_bbox', 'flipped_kp2d', 'init_kp3d', 'init_pose', 'flipped_init_kp3d', 'flipped_init_pose', 'global_orient', 'transl'])
